# SBS Detection Catalogue Inspection

First-pass inspection of the blendemu-built multi-shear detection catalogue. This notebook intentionally stops before classifier loading, prediction, or gradient extraction; those belong in the next notebook once the catalogue contents are understood.

In [ ]:
# Edit this cell to inspect a different catalogue.

SBS_ROOT = "/home/z/Zekang.Zhang/SBSI"
CATALOGUE = "/project/ls-gruen/users/zekang.zhang/lsst_selec_emu/sbs_detection_measurement_catalogue_train.feather"
FALLBACK_CATALOGUE = "/project/ls-gruen/users/zekang.zhang/lsst_selec_emu/detection_catalogue_train.feather"
RUN_LABEL = "FS2 / LSST selection-emulator detection catalogue"

# Keep interactive inspection bounded. Use Slurm for full-catalogue scans.
MAX_ROWS = 1_000_000
RANDOM_STATE = 321

RESCALE = dict(pixel_rms=0.312, pixel_size=0.2, zero_mag=30.0, psf_fwhm=0.73, moffat_beta=2.224)
PERCENTILES = [0.01, 0.05, 0.16, 0.5, 0.84, 0.95, 0.99]

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.ipc as ipc
import matplotlib.pyplot as plt

SBS_ROOT = Path(SBS_ROOT).expanduser().resolve()
if str(SBS_ROOT) not in sys.path:
    sys.path.insert(0, str(SBS_ROOT))

from sbsi.preprocessing import DEFAULT_CUTS, rescale, source_select_detection

plt.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "mathtext.fontset": "cm",
    "figure.dpi": 120,
})
cbPalette = ("#999999", "#E69F00", "#56B4E9", "#009E73", "#F0E442", "#0072B2", "#D55E00", "#CC79A7")

RAW_COLUMNS = [
    "input_index", "input_index_sec", "case", "shear_case",
    "detected", "neighbored", "distance",
    "Re_input_p", "Re_input_s",
    "r_input_p", "r_input_s",
    "sersic_n_input_p", "sersic_n_input_s",
    "axis_ratio_input_p", "axis_ratio_input_s",
    "redshift_input_p", "redshift_input_s",
    "gamma1_input_p", "gamma2_input_p", "gamma1_input_s", "gamma2_input_s",
    "e1_input_rot0_p", "e2_input_rot0_p", "e1_input_rot0_s", "e2_input_rot0_s",
    "polarization_angle", "shear_angle", "shear_component_convention",
]

MEASURED_COLUMNS = [
    "match_id_detec", "match_distance_pixel_cm", "match_dmag_cm",
    "measured_x_image", "measured_y_image", "measured_xwin_image", "measured_ywin_image",
    "measured_x_world", "measured_y_world",
    "measured_flux_auto", "measured_fluxerr_auto", "measured_mag_auto", "measured_magerr_auto",
    "measured_flux_aper", "measured_fluxerr_aper", "measured_mag_aper", "measured_magerr_aper",
    "measured_flux_radius", "measured_fwhm_image",
    "measured_a_image", "measured_b_image", "measured_a_world", "measured_b_world",
    "measured_theta_image", "measured_theta_world",
    "measured_flags", "measured_class_star", "measured_isoarea_image",
]

NUMERIC_RAW_FEATURES = [
    "r_input_p", "r_input_s",
    "Re_input_p", "Re_input_s",
    "sersic_n_input_p", "sersic_n_input_s",
    "axis_ratio_input_p", "axis_ratio_input_s",
    "redshift_input_p", "redshift_input_s",
    "distance",
    "gamma1_input_p", "gamma2_input_p", "gamma1_input_s", "gamma2_input_s",
    "e1_input_rot0_p", "e2_input_rot0_p", "e1_input_rot0_s", "e2_input_rot0_s",
]

SCALED_FEATURES = [
    "r_input_p_scaled", "r_input_s_scaled",
    "Re_input_p_scaled", "Re_input_s_scaled",
    "distance_scaled", "flux_ratio",
]


def open_feather_reader(path):
    return ipc.open_file(str(Path(path).expanduser().resolve()))


def read_catalogue_sample(path, columns, max_rows):
    path = Path(path).expanduser().resolve()
    with open_feather_reader(path) as reader:
        available = set(reader.schema.names)
        read_columns = [name for name in columns if name in available]
        missing = sorted(set(columns) - available)
        if missing:
            print(f"Columns absent from catalogue and ignored: {missing}")
        tables = []
        rows = 0
        for batch_index in range(reader.num_record_batches):
            table = pa.Table.from_batches([reader.get_batch(batch_index)]).select(read_columns)
            tables.append(table)
            rows += table.num_rows
            if max_rows and rows >= max_rows:
                break
        table = pa.concat_tables(tables) if tables else pa.table({name: [] for name in read_columns})
        if max_rows:
            table = table.slice(0, max_rows)
        return table.to_pandas()


def describe_frame(frame, columns, percentiles=PERCENTILES):
    columns = [name for name in columns if name in frame.columns]
    return frame[columns].describe(percentiles=percentiles).T


def boolean_rate_table(frame, group_cols):
    group_cols = [name for name in group_cols if name in frame.columns]
    if not group_cols:
        return pd.DataFrame()
    return (
        frame.groupby(group_cols, dropna=False)
        .agg(
            rows=("detected", "size"),
            detected_rate=("detected", "mean"),
            neighbored_fraction=("neighbored", "mean"),
            mean_distance=("distance", "mean"),
            mean_primary_mag=("r_input_p", "mean"),
            mean_neighbor_mag=("r_input_s", "mean"),
            mean_primary_z=("redshift_input_p", "mean"),
            mean_neighbor_z=("redshift_input_s", "mean"),
        )
        .reset_index()
    )


def binned_rate(frame, column, bins):
    values = []
    x = frame[column].to_numpy()
    y = frame["detected"].astype(float).to_numpy()
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (x >= lo) & (x < hi)
        values.append({
            "bin_center": 0.5 * (lo + hi),
            "rows": int(mask.sum()),
            "detected_rate": float(np.mean(y[mask])) if mask.any() else np.nan,
        })
    return pd.DataFrame(values)

## 1. File Metadata

In [ ]:
catalogue_path = Path(CATALOGUE).expanduser().resolve()
if not catalogue_path.exists() and FALLBACK_CATALOGUE:
    fallback_path = Path(FALLBACK_CATALOGUE).expanduser().resolve()
    print(f"Requested measured catalogue is not present yet: {catalogue_path}")
    print(f"Falling back to: {fallback_path}")
    catalogue_path = fallback_path

print(f"Run: {RUN_LABEL}")
print(f"Catalogue: {catalogue_path}")
print(f"File size: {catalogue_path.stat().st_size / 1024**3:.2f} GiB")

with open_feather_reader(catalogue_path) as reader:
    schema = reader.schema
    print(f"Record batches: {reader.num_record_batches:,}")
    print(f"Columns: {len(schema.names):,}")
    schema_table = pd.DataFrame({
        "column": schema.names,
        "type": [str(schema.field(name).type) for name in schema.names],
    })

display(schema_table)

## 2. Load Bounded Sample

In [ ]:
raw = read_catalogue_sample(catalogue_path, RAW_COLUMNS + MEASURED_COLUMNS, MAX_ROWS)
print(f"Loaded sample rows: {len(raw):,}")
print(f"Loaded columns: {len(raw.columns):,}")
print(f"Memory footprint: {raw.memory_usage(deep=True).sum() / 1024**2:.1f} MiB")

measured_present = [column for column in MEASURED_COLUMNS if column in raw.columns]
print(f"Measured columns present: {len(measured_present):,}")

display(raw.head())
display(raw.dtypes.rename("dtype").to_frame())

## 3. Basic Integrity Checks

In [ ]:
nulls = raw.isna().sum().rename("null_count").to_frame()
nulls["null_fraction"] = nulls["null_count"] / max(len(raw), 1)
finite = pd.Series(index=raw.columns, dtype=float, name="finite_fraction")
for column in raw.columns:
    if pd.api.types.is_numeric_dtype(raw[column]):
        finite.loc[column] = np.isfinite(raw[column].to_numpy()).mean()

display(nulls.join(finite).sort_values(["null_fraction", "finite_fraction"], ascending=[False, True]).head(20))

index_cols = [name for name in ["input_index", "input_index_sec", "case", "shear_case"] if name in raw.columns]
if index_cols:
    duplicate_count = int(raw.duplicated(index_cols).sum())
    print(f"Duplicate rows on {index_cols}: {duplicate_count:,}")

if {"input_index", "case", "shear_case"}.issubset(raw.columns):
    print(f"Unique primaries in sample: {raw['input_index'].nunique():,}")
    print(f"Unique primary/case/shear rows: {raw[['input_index', 'case', 'shear_case']].drop_duplicates().shape[0]:,}")

## 4. Shear And Case Coverage

In [ ]:
if "shear_case" in raw.columns:
    display(boolean_rate_table(raw, ["shear_case"]).sort_values("shear_case"))

if {"case", "shear_case"}.issubset(raw.columns):
    case_shear = pd.crosstab(raw["case"], raw["shear_case"])
    print(f"Cases represented in sample: {case_shear.shape[0]:,}")
    display(case_shear.head(20))

shear_cols = [name for name in ["gamma1_input_p", "gamma2_input_p", "gamma1_input_s", "gamma2_input_s"] if name in raw.columns]
if shear_cols:
    display(describe_frame(raw, shear_cols))

## 5. Detection And Blending Balance

In [ ]:
print(f"Detected fraction:  {raw['detected'].mean():.4f}")
print(f"Neighbored fraction: {raw['neighbored'].mean():.4f}")

display(boolean_rate_table(raw, ["neighbored"]))

if {"neighbored", "shear_case"}.issubset(raw.columns):
    display(boolean_rate_table(raw, ["shear_case", "neighbored"]).sort_values(["shear_case", "neighbored"]))

## 6. Apply SBS Detection Cuts And Rescaling

In [ ]:
selected = source_select_detection(raw, cuts=DEFAULT_CUTS)
scaled = rescale(selected, **RESCALE)
print(f"Rows before cuts: {len(raw):,}")
print(f"Rows after cuts:  {len(selected):,}")
print(f"Kept fraction:    {len(selected) / max(len(raw), 1):.4f}")
print(f"Detected fraction after cuts:  {selected['detected'].mean():.4f}")
print(f"Neighbored fraction after cuts: {selected['neighbored'].mean():.4f}")

if len(selected):
    display(boolean_rate_table(selected, ["neighbored"]))
    display(describe_frame(selected, NUMERIC_RAW_FEATURES))
    display(describe_frame(scaled, SCALED_FEATURES))

## 7. Measured Detection Properties

In [ ]:
measured_present = [column for column in MEASURED_COLUMNS if column in selected.columns]
if measured_present:
    print(f"Measured columns present after cuts: {len(measured_present):,}")
    measured_finite = selected[measured_present].notna().mean().rename("finite_fraction").to_frame()
    display(measured_finite.sort_values("finite_fraction"))
    display(describe_frame(selected, measured_present))
else:
    print("No measured-property columns in this catalogue. Build the SBS measured catalogue with blendemu/scripts/build_detection_catalogue.py --include-measured.")

## 8. Feature Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
plot_specs = [
    ("r_input_p", "primary r magnitude"),
    ("r_input_s", "neighbor r magnitude"),
    ("Re_input_p", "primary Re"),
    ("Re_input_s", "neighbor Re"),
    ("distance", "neighbor distance"),
    ("redshift_input_s", "neighbor redshift"),
]
for ax, (column, label) in zip(axes.ravel(), plot_specs):
    if column not in selected.columns:
        ax.set_visible(False)
        continue
    values = selected[column].to_numpy()
    values = values[np.isfinite(values)]
    ax.hist(values, bins=60, histtype="step", linewidth=2, color=cbPalette[5])
    ax.set_xlabel(label)
    ax.set_ylabel("count")
    ax.set_title(label)
plt.tight_layout()

## 9. Detection Rate Curves

In [ ]:
curve_specs = [
    ("r_input_p", np.linspace(18, 28, 26), "primary r magnitude"),
    ("r_input_s", np.linspace(18, 28, 26), "neighbor r magnitude"),
    ("distance", np.linspace(0, 5, 21), "neighbor distance"),
]

fig, axes = plt.subplots(1, len(curve_specs), figsize=(5 * len(curve_specs), 4))
if len(curve_specs) == 1:
    axes = [axes]
for ax, (column, bins, label) in zip(axes, curve_specs):
    if column not in selected.columns:
        ax.set_visible(False)
        continue
    rates = binned_rate(selected, column, bins)
    good = rates["rows"] > 0
    ax.plot(rates.loc[good, "bin_center"], rates.loc[good, "detected_rate"], marker="o", linewidth=2, color=cbPalette[6])
    ax.set_xlabel(label)
    ax.set_ylabel("detected fraction")
    ax.set_ylim(-0.05, 1.05)
    ax.set_title(label)
plt.tight_layout()

## 10. Redshift-Aware Blend Structure

In [ ]:
if {"redshift_input_p", "redshift_input_s"}.issubset(selected.columns):
    zview = selected.copy()
    zview["delta_z_s_minus_p"] = zview["redshift_input_s"] - zview["redshift_input_p"]
    zview["abs_delta_z"] = zview["delta_z_s_minus_p"].abs()
    display(describe_frame(zview, ["redshift_input_p", "redshift_input_s", "delta_z_s_minus_p", "abs_delta_z"]))

    dz_bins = np.linspace(np.nanpercentile(zview["delta_z_s_minus_p"], 1), np.nanpercentile(zview["delta_z_s_minus_p"], 99), 25)
    rates = binned_rate(zview, "delta_z_s_minus_p", dz_bins)
    fig, ax = plt.subplots(figsize=(7, 4))
    good = rates["rows"] > 0
    ax.plot(rates.loc[good, "bin_center"], rates.loc[good, "detected_rate"], marker="o", linewidth=2, color=cbPalette[5])
    ax.axvline(0, color="k", linestyle="--", linewidth=1, alpha=0.5)
    ax.set_xlabel("neighbor redshift - primary redshift")
    ax.set_ylabel("detected fraction")
    ax.set_ylim(-0.05, 1.05)
    ax.set_title("Detection rate by redshift separation")
    plt.tight_layout()

## 11. Notes For Next Step

After this catalogue inspection is satisfactory, add a separate classifier inspection notebook or extend this notebook with model loading, predicted detection probabilities, calibration diagnostics, and differentiable shear-gradient checks. Full-catalogue summaries should be run as Slurm jobs rather than from an interactive notebook.